In [ ]:
import pandas as pd 
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
plt.rcParams['figure.dpi'] = 72 

In [ ]:
o_df = pd.read_csv('../data/processed/smartphones_cleaned.csv')

#### GOAL ?
  - to identify the hidden pattern lying inside the data
  - to understand the effect of other parameters on price
  - to extract meaningful insights from data

## Points
  - why single sim are so costlier ? <br>
  <b>ANS : </b> because single sim is led primarily by apple iphones and some of the foldable premium phones so that's why their prices are higher <br>
  <b>Added : </b> only 6 phones are single sim (iPhone 8, iPhone 8 Plus, Galaxy Z Flip 5, Motorola Razr 2019), so this is a very small group. It is not a real trend

  - [Vo5G, NFC, eSIM] have [weak, strong, weak] correlation with price respectively <br>
  <b>Correction : </b> when I check the actual numbers all three go with a higher price (median price with vs without: Vo5G 56k vs 25k, NFC 45k vs 19k, eSIM 77k vs 25k). eSIM has the biggest gap but only 131 phones have it, NFC is the one that most phones have (397), so NFC is the most useful of the three

  - as the number of cores is increasing the price is also increasing but the one number does not fit in linearity that is 6 cores also 1 & 2 cores processors have price greater than 4 why ? <br>
  <b> ANS: </b> 6 cores is used by apple iphones (all 31 hexa-core phones are iPhones). 1 and 2 core phones have only 7 and 1 phones and 4 core has only 6 old cheap phones, so these groups are too small to trust

  - why 2gb ram phones show a higher average price than 3gb and 4gb ? <br>
  <b> ANS: </b> because apple iphone 8 (price 77000) is in the 2gb group, and there are only 7 phones in that group so the mean was pulled up. The median price goes in the right order (7.5k, 10.4k, 14k) <br>
  <b>Added : </b> same reason for 18gb < 16gb, there is only one 18gb phone (ASUS ROG Phone 5s Pro, 79,999)

  - why there are no battery values between 8000 - 9000 and between 9020 - 10001 ? <br>
  <b> ANS: </b> only 34 phones are above 8000 mAh and they come at only 4 values (8000, 9000, 9020, 10001). These are new big batteries and companies launch them with round numbers, so nothing is in between. It can also be that the dataset is missing some phones, which we cannot confirm from this data

  - why are some rows empty in the correlation heatmap (Vo5G, NFC, eSIM, IR_Blaster, isFoldable, isDualDisplay) ? <br>
  <b> ANS: </b> these columns have only 1 or NaN, so correlation cannot be calculated. They need fillna(0) first (done in the multivariate section)

  - why Bionic and Apple were counted as two different processor brands ? <br>
  <b> ANS: </b> replace() only changes the exact word 'Bionic' but names like 'Bionic A16' kept the word after the split. Fixed in the cleaning part

### Insights
  - do you know smartphones that have no memory card_suport are often costlier !! Because "no memory card slot" is a premium design choice, not a downgrade <br>
  - price is right skewed (mean - 37,798 & median - 26,999), a few very costly phones (foldables, iphones) pull the mean up, so median is better for overall price representation
  - clock speed, storage and ram are the features most linked with price. Rating and battery are almost not linked with price
  - Apple is the costliest, and with the same ram and storage an iphone can still cost about 2.5x of an android phone 
  - refresh rate: 90hz is cheap, 120hz and 144hz have the same median price, only 165hz (gaming phones) costs more
  - display design tells the tier clearly: water drop notch < punch hole < dynamic island (iphone uses always)
  - camera megapixels is a weak signal, 50mp phones are very common (562 phones), only 108mp+ phones are really costly
  - flagship phones have smaller batteries than mid range phones

In [ ]:
o_df.columns

In [ ]:
# inspecting the data
o_df.info()
o_df.dtypes
o_df.describe().round(4)

After inspecting the data it has been observed that some of the columns can not be of any help as a result they have been discarded

In [ ]:
original_columns = o_df.columns
excluded_columns = ['index', 'sim_original', 'Processor', 'Ram', 'Battery', 'Display', 'Camera', 'Card', 'Os', 'VoLTE', 'Wi-Fi']
new_columns = []
for col in original_columns:
    if not col in excluded_columns:
        new_columns.append(col)
df = o_df[new_columns].copy()   # .copy() added, this removes the SettingWithCopyWarning from the next cell

Observation shows that some columns had dtype fundamentally wrong so they were converted, and columns like processor_name can be of better help if it were processor_brand so, it has to be converted

In [ ]:
temp = ['battery(mAh)', 'charging_support(W)', 'rear_main_camera'] # columns whose dtype is wrong

df['Network'] = df['Network'].str.strip().str.split("G").str[0].str.strip().astype('Int64')
df.loc[df['charging_support(W)'] == 'Unknown', 'charging_support(W)'] = np.nan
df.loc[df['battery(mAh)'] == 'Unknown', 'battery(mAh)'] = np.nan
df.loc[df['rear_main_camera'] == '\n', 'rear_main_camera'] = np.nan

# change the dtypes
for col in temp:
    if col == 'rear_main_camera':
        df[col] = df[col].astype('Float64', errors='ignore')
    else :
        df[col] = df[col].astype('Int64')
        
        
processor_cores_map = {
    'Octa Core':8,
    'Hexa Core':6,
    'Single Core':1,
    'Deca Core':10,
    'Nine Cores':9,
    'Quad Core':4,
    'Dual Core':2,
}
df['processor_cores'] = df['processor_cores'].replace(processor_cores_map)

df['processor_name'] = df['processor_name'].str.strip().str.title()
replace_map = {
    'Unknkown': 'Unknown',
    'Tiger': 'Unisoc',
    'Mtk6739': 'Mediatek',
    'Mt8125': 'Mediatek',
    'Mtk6763': 'Mediatek',
    'Bionic': 'Apple',
}
df['processor_name'] = df['processor_name'].replace(replace_map)
df['processor_name'] = df['processor_name'].replace('Unknown', np.nan)
df['processor_name'] = df['processor_name'].str.split(" ").str[0]

df = df.rename(columns={
    'processor_name':'processor_brand'
})

x = df.dtypes.reset_index()
object_columns=x[x[0] == 'object']['index']
df[object_columns]

df = df[~(df.index.isin([625, 759]))]

df.loc[df['display_design'] == ' Dynamic Island Display', 'display_design'] = ' Dynamic Island'

df = df.drop(columns=['rear_total_mp', 'front_total_mp'])

In [ ]:
before = df['processor_brand'].value_counts()
df['processor_brand'] = df['processor_brand'].replace('Bionic', 'Apple')
after = df['processor_brand'].value_counts()

pd.concat([before, after], axis=1, keys=['before', 'after']).fillna(0).astype(int)

In [ ]:
df['processor_cores'] = pd.to_numeric(df['processor_cores'])

In [ ]:
missing = pd.concat([df.isna().sum(), (df.isna().mean() * 100).round(1)], axis=1, keys=['missing', 'percent'])
missing[missing['missing'] > 0].sort_values('missing', ascending=False)

## UNIVARIATE ANALYSIS

In [ ]:
print(df['Price'].describe())
sns.histplot(data=df, x='Price', kde=True)
plt.show()
print("Skewness: ", df['Price'].skew())
sns.boxplot(data=df, x='Price')
plt.show()

df[df['Price']>200000]

In [ ]:
print(df['Specs'].describe())
print("skewness: ", df['Specs'].skew().round(4)) # type: ignore
sns.displot(kind='hist', kde=True, data=df,x='Specs')
plt.show()
sns.boxplot(data=df, x='Specs')
# df = df[~(df['Specs']<25)]

In [ ]:
df[df['Specs'] < 25][['Name', 'Price', 'Specs', 'Brand']]

**Points noticed :** Specs is a score given by the website, so it is a derived column as a result they will not be used for any kind of analysis because in real scenario you are not provided with specs like score when buying a phone (usually, except for some processor performance score entity) 

In [ ]:
print(df['Rating'].describe())
print("skewness: ", df['Rating'].skew().round(4)) # type: ignore
sns.displot(kind='hist', kde=True, data=df,x='Rating')
plt.show()
sns.boxplot(data=df, x='Rating')

In [ ]:
df['Brand'].value_counts()
# df['Brand'].value_counts().head(10).plot(kind='pie', autopct='%0.1f%%')
# plt.show()
sns.countplot(x=df['Brand'])
plt.xticks(rotation='vertical')
plt.show()

In [ ]:
pd.concat([df['Brand'].value_counts(), df['Brand'].value_counts(normalize=True).mul(100).round(1)],
          axis=1, keys=['phones', 'percent']).head(10)

`df['SIM']` on its own prints the whole column, so it is left commented. `value_counts` below tells the same thing.

In [ ]:
df['SIM'].value_counts()
# df['SIM'].value_counts().head(10).plot(kind='pie', autopct='%0.1f%%')
# plt.show()
sns.countplot(x=df['SIM'])
plt.xticks(rotation='vertical')
plt.show()

In [ ]:
df['Network'].value_counts()
# df['Network'].value_counts().head(10).plot(kind='pie', autopct='%0.1f%%')
# plt.show()
sns.countplot(x=df['Network'])
plt.xticks(rotation='vertical')
plt.show()

In [ ]:
x = df['NFC'].fillna(0)
sns.histplot(x=x, y=df['Price'])

In [ ]:
x = df['eSIM'].fillna(0)
sns.histplot(x=x, y=df['Price'])

In [ ]:
x = df['IR_Blaster'].fillna(0)
sns.histplot(x=x, y=df['Price'])

In [ ]:
x = df['Vo5G'].fillna(0)
sns.histplot(x=x, y=df['Price'])
plt.show()

<b> NOTE :</b> Some of the processor_brand does not have naming based on their chipset family series most probably because of unpopularity of their chipset family 

In [ ]:
df['processor_brand'].value_counts()
sns.countplot(data=df, x='processor_brand', order=df['processor_brand'].value_counts().index)
plt.xticks(rotation='vertical')
plt.show()
df['processor_brand'].value_counts()

In [ ]:
df['processor_cores'].value_counts()
sns.countplot(data=df, x='processor_cores')
sns.barplot(data=df, x='processor_cores', y='Price')

the uprising in prices for the phones having processor_cores == 6 is because of apple. Apple often use 6 cores processors

In [ ]:
df['processor_clock_speed(GHz)'].value_counts()

plt.figure(figsize=(14, 5))
sns.countplot(data=df, x='processor_clock_speed(GHz)')
plt.xticks(rotation='vertical')
plt.show()

sns.histplot(data=df, x='processor_clock_speed(GHz)', kde=True)
plt.show()

sns.boxplot(data=df, x='processor_clock_speed(GHz)')
plt.show()
print(df['processor_clock_speed(GHz)'].skew())

df['processor_clock_speed(GHz)'].describe()

df[df['processor_clock_speed(GHz)'] > 4]['Price'].describe()

In [ ]:
df['ram(GB)'].value_counts()
sns.countplot(data=df, x='ram(GB)')
plt.show()
# df['ram(GB)'].value_counts().head(7).plot(kind='pie', autopct='%0.1f%%')
# plt.show()

# df.select_dtypes(['float', 'int']).corr()['Price'].loc['ram(GB)']
sns.barplot(data=df, x='ram(GB)', y='Price')

**Insight :** 2 gb ram phones have price greater than of 4 is because of apple iphone 8 plus that cost around 77k so it shifted mean and as a result 2gb ram phones seems to have higher pricer than of 2gb ram phones, same case with 16gb ram phones can be explained, it's because so many foldable phones gives 16gb ram that costs more than 2 lakhs, on the other hand there is only one phone(oppo find x9 ultra) that offers 18gb ram with price around ~ 80K 

In [ ]:
df['storage(GB)'].value_counts()
sns.countplot(data=df, x='storage(GB)')
plt.show()
# df['storage(GB)'].value_counts().head(7).plot(kind='pie', autopct='%0.1f%%')
# plt.show()

print(df['Price'].corr(df['storage(GB)']))
sns.barplot(data=df, x='storage(GB)', y='Price')
plt.show()
df[df['storage(GB)'] == 2000]

In [ ]:
df['battery(mAh)'].value_counts()
df['battery(mAh)'].describe()
sns.histplot(data=df, x='battery(mAh)', kde=True)
plt.show()
sns.boxplot(data=df, x='battery(mAh)')
plt.show()
df[df['battery(mAh)'] > 8000]

In [ ]:
sns.histplot(data=df, x='charging_support(W)', kde=True)
plt.show()
print(df['charging_support(W)'].skew())
df['charging_support(W)'].value_counts().sort_index()
df['charging_support(W)'].describe()
# plt.figure(figsize=(14, 6))

In [ ]:
print("display_size median:", df['display_size'].median(), "\tdisplay_size mode-", df['display_size'].mode()[0])
print(df['display_size'].describe())
sns.boxplot(data=df, x='display_size')
plt.show()
df[df['display_size'] == df['display_size'].min()]
sns.kdeplot(data=df, x='display_size')
plt.show()
print(df['display_size'].skew())

In [ ]:
# display_resolution will be considered later 

In [ ]:
df['refresh_rate(Hz)'].value_counts().sum()
sns.countplot(data=df, x='refresh_rate(Hz)')
plt.show()

In [ ]:
df['display_design'].value_counts()
df[df['display_design'].isna()][['Name', 'display_design', 'Price']]
sns.countplot(data=df, x='display_design')
plt.xticks(rotation='vertical')
plt.show()

In [ ]:
print(df['isFoldable'].value_counts())
print(df['isDualDisplay'].value_counts())

In [ ]:
df['rear_main_camera'].value_counts().sum() 
sns.countplot(data=df, x='rear_main_camera')
plt.show()
# df['rear_main_camera'].value_counts().head(10).plot(kind='pie', autopct='%0.1f%%')
# plt.show()

df['rear_main_camera'].describe()
sns.histplot(data=df, x='rear_main_camera', kde=True)
plt.show()
sns.boxplot(data=df, x='rear_main_camera')

In [ ]:
df['rear_camera_count'].value_counts()
sns.countplot(data=df, x='rear_camera_count')
plt.show()
# df['rear_camera_count'].value_counts().plot(kind='pie', autopct='%0.1f%%')

In [ ]:
df['front_main_camera'].value_counts().sum() 
sns.countplot(data=df, x='front_main_camera')
plt.xticks(rotation='vertical')
plt.show()
# df['front_main_camera'].value_counts().head(10).plot(kind='pie', autopct='%0.1f%%')
# plt.show()

df['front_main_camera'].describe()
sns.histplot(data=df, x='front_main_camera', kde=True)
plt.show()
df['front_main_camera'].plot(kind='box')

In [ ]:
df['front_camera_count'].value_counts()
sns.countplot(data=df, x='front_camera_count')
plt.show()
# df['front_camera_count'].value_counts().plot(kind='pie', autopct='%0.1f%%')

In [ ]:
df['card_supported'].value_counts()
sns.countplot(data=df, x='card_supported')

In [ ]:
df['card_type'].value_counts()
sns.countplot(data=df, x='card_type')

In [ ]:
df['card_capacity(GB)'].value_counts()
sns.countplot(data=df, x='card_capacity(GB)')
plt.show()
# df['card_capacity(GB)'].value_counts().plot(kind='pie', autopct='%0.1f%%')

## BIVARIATE ANALYSIS

In [ ]:
# name can't be of any help so we'll ignore that also in real world specs are not given they are derived parameters so that will also not be considered, same goes for rating column

In [ ]:
sns.barplot(data=df, x='Brand', y='Price')
plt.xticks(rotation='vertical')
plt.show()

sns.barplot(data=df, x='SIM', y='Price')
plt.xticks(rotation='vertical')
plt.show()

df[df['SIM'] == 'Single Sim']

**NOTE:**  single sim smartphones are offered by either apple or foldable premium phones. that's why thier is much difference between mean of dual sim smartphones and sigle sim smartphones 

In [ ]:
brand_price = df.groupby('Brand')['Price'].agg(['count', 'mean', 'median', 'max'])
brand_price[brand_price['count'] >= 10].sort_values('median', ascending=False).round(0)

In [ ]:
sns.barplot(data=df, x='Network', y='Price')
plt.xticks(rotation='vertical')
plt.show()

**Insight :** as we all know 5g phones most usually costlier than 4g but do you know ? only 5G label does not make a smartphone completely 5g because while calling 5G may switch to 4G lines resulting in lower internet and connection speed but if your smartphone have Vo5G feature than it shall continue to 5G lines.

In [ ]:
x = df['Vo5G'].fillna(0)
sns.barplot(x=x, y=df['Price'])
plt.show()

In [ ]:
x = df['NFC'].fillna(0)
sns.barplot(x=x, y=df['Price'])
plt.show()

<b> NOTE : </b> for those who don't know NFC feature - NFC (Near Field Communication) is a short-range wireless technology that allows smartphones to exchange data with other devices or tags when held within 4 centimeters (about 1.5 inches) of each other.  It operates using electromagnetic induction, enabling powered devices (like phones) to communicate with passive tags or readers without needing a battery in the target device.

In [ ]:
x = df['eSIM'].fillna(0)
sns.barplot(x=x, y=df['Price'])
plt.show()
# df[df['NFC'] == 1]

In [ ]:
x = df['IR_Blaster'].fillna(0)
sns.barplot(x=x, y=df['Price'])
plt.show()

<b> NOTE: </b> IR_Blaster -- An IR blaster is a hardware component embedded in certain smartphones that emits invisible infrared signals to control electronic devices, effectively turning the phone into a universal remote.  It works by transmitting specific infrared light pulses that mimic traditional remote controls, allowing users to operate TVs, air conditioners, DVD players, and set-top boxes without needing Wi-Fi, Bluetooth pairing, or an internet connection

Some of the processor_brand does not have naming based on their chipset family series most probably because of unpopularity of their chipset family

In [ ]:
# sns.scatterplot(data=df, x='processor_brand', y='Price')
# plt.xticks(rotation='vertical')
# plt.show()

sns.barplot(data=df, x='processor_brand', y='Price')
plt.xticks(rotation='vertical')
plt.show()

sns.boxplot(data=df, x='processor_brand', y='Price')
plt.xticks(rotation='vertical')
plt.show()

In [ ]:
df.groupby('processor_brand').agg(phones=('Price', 'size'),
                                  median_price=('Price', 'median'),
                                  avg_clock=('processor_clock_speed(GHz)', 'mean')).round(2).sort_values('median_price', ascending=False)

In [ ]:
# sns.scatterplot(data=df, x='processor_cores', y='Price')
# plt.xticks(rotation='vertical')
# plt.show()

sns.barplot(data=df, x='processor_cores', y='Price')
plt.xticks(rotation='vertical')
plt.show()

sns.boxplot(data=df, x='processor_cores', y='Price')
plt.xticks(rotation='vertical')
plt.show()

df[df['processor_cores'] < 6][['Name', 'Price', 'ram(GB)', 'storage(GB)', 'processor_brand', 'processor_cores']]

In [ ]:
df[df['processor_cores'] == 6]['Brand'].value_counts()

In [ ]:
sns.scatterplot(data=df, x='processor_clock_speed(GHz)', y='Price')
plt.show()

print(df['Price'].corr(df['processor_clock_speed(GHz)']))

sns.histplot(data=df,x='Price', y='processor_clock_speed(GHz)', cbar=True)
plt.show()

# sns.kdeplot(data=df,x='Price', y='processor_clock_speed(GHz)', cbar=True)
# plt.show()

In [ ]:
sns.scatterplot(data=df, x='ram(GB)', y='Price')
plt.xticks(rotation='vertical')
plt.show()

sns.barplot(data=df, x='ram(GB)', y='Price')
plt.xticks(rotation='vertical')
plt.show()

sns.boxplot(data=df, x='ram(GB)', y='Price')
plt.xticks(rotation='vertical')
plt.show()

x = df[df['ram(GB)'] == 2]
display(x[x['Price'] > df[df['ram(GB)'] == 4]['Price'].min()][['Name', 'Price', 'ram(GB)']])

x = df[(df['ram(GB)'] == 16)]
display(x[x['Price'] > df[df['ram(GB)'] == 18]['Price'].values[0]][['Name', 'Price', 'ram(GB)']])

In [ ]:
pd.concat([df.groupby('ram(GB)')['Price'].mean(), df.groupby('ram(GB)')['Price'].median(), df['ram(GB)'].value_counts()],
          axis=1, keys=['mean', 'median', 'phones']).round(0)

In [ ]:
# sns.scatterplot(data=df, x='storage(GB)', y='Price')
# plt.xticks(rotation='vertical')
# plt.show()

sns.barplot(data=df, x='storage(GB)', y='Price')
plt.xticks(rotation='vertical')
plt.show()

sns.boxplot(data=df, x='storage(GB)', y='Price')
plt.xticks(rotation='vertical')
plt.show()

print(df['Price'].corr(df['storage(GB)']))

In [ ]:
pd.concat([df.groupby('storage(GB)')['Price'].mean(), df.groupby('storage(GB)')['Price'].median(), df['storage(GB)'].value_counts()],
          axis=1, keys=['mean', 'median', 'phones']).round(0)

In [ ]:
sns.scatterplot(data=df, x='battery(mAh)', y='Price')
plt.show()

print(df['Price'].corr(df['battery(mAh)']))

sns.histplot(data=df,x='Price', y='battery(mAh)', cbar=True)
plt.show()

# sns.kdeplot(data=df,x='Price', y='battery(mAh)', cbar=True)
# plt.show()

In [ ]:
df.groupby(pd.qcut(df['Price'], 5))['battery(mAh)'].median()

In [ ]:
df['battery(mAh)'].value_counts().sort_index().tail(10)

**Insight :** battery has almost no link with price, instead it has slightly -ve correlation(~ -14) it's because the budget price phones provides more battery capacity as marketing parameter but premium phones still intact with mostly 5000-7000 mAh batteries 

In [ ]:
sns.scatterplot(data=df, x='display_size', y='Price')
plt.show()

print(df['Price'].corr(df['display_size']))

sns.histplot(data=df,x='Price', y='display_size', cbar=True)
plt.show()

# sns.kdeplot(data=df,x='Price', y='display_size', cbar=True)
# plt.show()

print(df['Price'].corr(df['display_size']))

In [ ]:
df.sort_values('display_size', ascending=False)[['Name', 'display_size', 'Price']].head(5)

In [ ]:
# # sns.scatterplot(data=df, x='refresh_rate(Hz)', y='Price')
# # plt.xticks(rotation='vertical')
# # plt.show()

sns.barplot(data=df, x='refresh_rate(Hz)', y='Price')
plt.xticks(rotation='vertical')
plt.show()

sns.boxplot(data=df, x='refresh_rate(Hz)', y='Price')
plt.xticks(rotation='vertical')
plt.show()

df[(df['refresh_rate(Hz)'] >= 60)  & (df['Brand'] == 'Apple')]
x = df[(df['refresh_rate(Hz)'] == 144)]
x[x['Price'] > df[df['refresh_rate(Hz)'] > 144]['Price'].max()]

In [ ]:
df.groupby('refresh_rate(Hz)')['Price'].agg(['count', 'mean', 'median']).round(0)

In [ ]:
# sns.scatterplot(data=df, x='display_design', y='Price')
# plt.xticks(rotation='vertical')
# plt.show()

sns.barplot(data=df, x='display_design', y='Price')
plt.xticks(rotation='vertical')
plt.show()

sns.boxplot(data=df, x='display_design', y='Price')
plt.xticks(rotation='vertical')
plt.show()

# sns.violinplot(data=df, x='display_design', y='Price')
# plt.xticks(rotation='vertical')
# plt.show()

In [ ]:
df.groupby('display_design')['Price'].agg(['count', 'median']).sort_values('median')

In [ ]:
d = df[df['Brand'] != 'Apple']
x = d['isFoldable'].fillna(0)

sns.barplot(x=x, y=d['Price'])
plt.xticks()
plt.show()

sns.boxplot(x=x, y=d['Price'])
plt.xticks()
plt.show()

# sns.scatterplot(x=x, y=d['Price'])
# sns.violinplot(x=x, y=d['Price'])

In [ ]:
# x = df['isDualDisplay'].fillna(0)
# sns.scatterplot(x=x, y=df['Price'])
# plt.xticks()
# plt.show()

sns.barplot(x=x, y=df['Price'])
plt.xticks()
plt.show()

sns.boxplot(x=x, y=df['Price'])
plt.xticks()
plt.show()

# sns.violinplot(x=x, y=df['Price'])
# plt.xticks()
# plt.show()

In [ ]:
sns.scatterplot(data=df, x='rear_main_camera', y='Price')
plt.show()

print(df['Price'].corr(df['rear_main_camera']))

sns.barplot(data=df, x='rear_main_camera', y='Price')
plt.xticks(rotation='vertical')
plt.show()

sns.boxplot(data=df, x='rear_main_camera', y='Price')
plt.xticks(rotation='vertical')
plt.show()

df[(df['rear_main_camera'] > 13) & (df['rear_main_camera'] < 24)]

In [ ]:
# sns.scatterplot(data=df, x='rear_camera_count', y=df['Price'])
# plt.xticks()
# plt.show()

sns.barplot(data=df, x='rear_camera_count', y=df['Price'])
plt.xticks()
plt.show()

sns.boxplot(data=df, x='rear_camera_count', y=df['Price'])
plt.xticks()
plt.show()

# sns.violinplot(data=df, x='rear_camera_count', y=df['Price'])
# plt.xticks()
# plt.show()

df[(df['rear_camera_count'] == 5)]

In [ ]:
# sns.scatterplot(data=df, x='front_main_camera', y='Price')
# plt.show()

print(df['Price'].corr(df['front_main_camera']))

plt.figure(figsize=(10, 5))
sns.barplot(data=df,x='front_main_camera', y='Price')
plt.show()

sns.boxplot(data=df,x='front_main_camera', y='Price')
plt.xticks(rotation='vertical')
plt.show()
print(df['Price'].corr(df['front_main_camera']))

In [ ]:
# sns.scatterplot(data=df, x='front_camera_count', y=df['Price'])
# plt.xticks()
# plt.show()

sns.barplot(data=df, x='front_camera_count', y=df['Price'])
plt.xticks()
plt.show()

sns.boxplot(data=df, x='front_camera_count', y=df['Price'])
plt.xticks()
plt.show()

# sns.violinplot(data=df, x='front_camera_count', y=df['Price'])
# plt.xticks()
# plt.show()

x = df[(df['front_camera_count'] > 1)]
x[x['Price'] > 150000]

In [ ]:
# sns.scatterplot(data=df, x='card_supported', y=df['Price'])
# plt.xticks()
# plt.show()

sns.barplot(data=df, x='card_supported', y=df['Price'])
plt.xticks()
plt.show()

sns.boxplot(data=df, x='card_supported', y=df['Price'])
plt.xticks()
plt.show()

# sns.violinplot(data=df, x='card_supported', y=df['Price'])
# plt.xticks()
# plt.show()

In [ ]:
# groupby : price with and without a card slot
df.groupby('card_supported')['Price'].agg(['count', 'mean', 'median']).round(0)

In [ ]:
# sns.scatterplot(data=df, x='card_type', y=df['Price'])
# plt.xticks()
# plt.show()

sns.barplot(data=df, x='card_type', y=df['Price'])
plt.xticks()
plt.show()

sns.boxplot(data=df, x='card_type', y=df['Price'])
plt.xticks()
plt.show()

# sns.violinplot(data=df, x='card_type', y=df['Price'])
# plt.xticks()
# plt.show()

In [ ]:
sns.scatterplot(data=df, x='card_capacity(GB)', y='Price')
plt.show()

print(df['Price'].corr(df['card_capacity(GB)']))

sns.barplot(data=df,y='Price', x='card_capacity(GB)')
plt.show()

sns.boxplot(data=df,y='Price', x='card_capacity(GB)')
plt.show()

print(df['Price'].corr(df['card_capacity(GB)']))
df[(df['card_capacity(GB)'] == 1000) & (df['Price'] > 90000)]

## MULTIVARIATE ANALYSIS

In [ ]:
corr = df.corr(numeric_only=True, method="spearman")
plt.figure(figsize=(30, 9))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.show()

**Points noticed (heatmap) :** the heatmap uses spearman correlation. The blank rows and columns are Vo5G, NFC, eSIM, IR_Blaster, isFoldable and isDualDisplay. They only have 1 or NaN, so there is nothing to calculate. Fixed in the next cell by using fillna(0) first.

In [ ]:
df2 = df.copy()
flags = ['Vo5G', 'eSIM', 'NFC', 'isFoldable', 'isDualDisplay', 'IR_Blaster']
df2[flags] = df2[flags].fillna(0)

corr = df2.corr(numeric_only=True, method="spearman")
plt.figure(figsize=(30, 9))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.show()

x = corr['Price'].sort_values(ascending=False).round(2)
sns.barplot(x=x.values, y=x.index, orient='h').set(title='Price correlation with all others columns', xlabel='Values', ylabel='Columns')

**Insight (heatmap) :**
  - biggest links with price: clock speed 0.83, ram 0.81, storage 0.80. Then NFC 0.64, charging 0.59, front camera mp 0.56, network (5G) 0.53 and eSIM 0.51.
  - Rating (0.02), processor_cores (0.02) and battery (-0.02) are almost zero, they do not explain price
  - card_supported is the only strong negative one (-0.72). It is even more negative with clock speed (-0.74), ram (-0.61) and storage (-0.60).
  - ram, storage and clock speed are also linked with each other. which is very self explanatory
  

In [ ]:

df['price_band'] = pd.cut(df['Price'], [0, 15000, 25000, 40000, 60000, 300000],
                          labels=['<15k', '15-25k', '25-40k', '40-60k', '60k+'])

band_profile = df.groupby('price_band', observed=True).agg(phones=('Price', 'size'),
                                            avg_ram=('ram(GB)', 'median'),
                                             avg_storage=('storage(GB)', 'median'),
                                             avg_clock=('processor_clock_speed(GHz)', 'median'),
                                             avg_battery=('battery(mAh)', 'median'),                                                 avg_charging=('charging_support(W)', 'median'))

nfc_pct = df.groupby('price_band', observed=True)['NFC'].apply(lambda s: s.fillna(0).mean() * 100).rename('NFC_%')
slot_pct = df.groupby('price_band', observed=True)['card_supported'].mean().mul(100).rename('card_slot_%')
pd.concat([band_profile, nfc_pct, slot_pct], axis=1).round(1)

**Insight (price groups) :** this table shows how a phone changes when the price goes up.
  - ram, storage and clock speed increases as price increases which is normal and so obvious
  - NFC% increases as price increases that made him a premium feature which is again very obvious
  - battery and charging do not keep rising: they are also linear with price(as price go higher the capacities go higher too), it's just there are foldable smartphones and dualDisplay smartphones that does not provide much better values because of the foldability concern

In [ ]:
df['card_status'] = df['card_supported'].map({1.0: 'Slot', 0.0: 'No slot'}).fillna('Unknown')

plt.figure(figsize=(9, 5))
sns.scatterplot(data=df, x='processor_clock_speed(GHz)', y='Price', hue='card_status',
                palette={'Slot': '#27ae60', 'No slot': '#c0392b', 'Unknown': '#bdc3c7'}, alpha=0.7)
plt.show()

**Insight (card slot vs clock speed) :** from the above graph it can be concluded that as fast as processor_speed will be the more chance will there be absence of card_slot, also if there are phones are providing card_slot that are costlier, those phones are older iphones providinig card_slot. but as of today's modern smartphones desgin choice shows that almost all premium phones does not provide card_slot

In [ ]:
pt = pd.pivot_table(df, values='Price', index='ram(GB)', columns='storage(GB)', aggfunc='median')
plt.figure(figsize=(10, 5))
sns.heatmap(pt, annot=True, fmt='.0f', cmap='YlGnBu')
plt.show()

In [ ]:
brand_summary = df.groupby('Brand').agg(phones=('Price', 'size'), median_price=('Price', 'median'),
                                        median_ram=('ram(GB)', 'median'), median_storage=('storage(GB)', 'median'))
nfc_share = df.groupby('Brand')['NFC'].apply(lambda s: s.fillna(0).mean() * 100).rename('NFC_%')
slot_share = df.groupby('Brand')['card_supported'].mean().mul(100).rename('card_slot_%')

brand_summary = pd.concat([brand_summary, nfc_share, slot_share], axis=1)
brand_summary[brand_summary['phones'] >= 15].sort_values('median_price', ascending=False).round(1)

**Insight** : it's worth noticing that brands like google and apple provides NFC feature 100% for sure at least for this dataset

In [ ]:
top_brands = df['Brand'].value_counts().head(10).index
x = pd.crosstab(df[df['Brand'].isin(top_brands)]['Brand'], df['processor_brand']) 

sns.heatmap(x, annot=True, fmt=".2f", cmap="coolwarm", center=0)

**Insight (crosstabs) :**
  - Exynos is used only by Samsung (67 phones, and Samsung also uses Snapdragon 64 and Dimensity 27). Realme and Oppo mostly use Dimensity (75 and 72 phones), while Xiaomi, Motorola and OnePlus mostly use Snapdragon
  - NFC is almost only a 5G thing: 51% of 5G phones have NFC but only 6% of 4G phones have it. So NFC, 5G and a higher price all come together